# Phase 3, step 3 — the agent writes its own features

Step 2 gave the agent `fit_encoding`, and with it a rule: *anything that has to
learn a parameter from the data goes through a structured tool*. That rule is
what keeps target leakage out. But it leaves an enormous amount of ordinary
feature engineering with nowhere to go.

`price / area`. `2026 - year_built`. `date.dt.month`. None of those learn
anything — they are arithmetic on one row — and none of them can be expressed as
an encoder. Writing a tool for each would never end.

So the fifth tool, `transform_features`, hands the agent a **Python sandbox**:
it writes `df -> df` code, we run it, and the resulting columns become a new
feature table. This notebook is about the two questions that makes urgent:

1. If the agent can run arbitrary pandas, what stops it from leaking?
2. The columns are computed *now* — how do they exist again at predict time?

As always: drive it **by hand** first, LLM only at the very end.

## 1. A dataset with a column the model cannot see

Apartments, and whether each one sold quickly. The strongest driver is **when it
was listed** — spring sells, December does not — and there is a weekend effect on
top. That information is in the data, in a column called `listed_at`.

It is also completely unavailable to the model. `listed_at` is a string, so
`train_tabular_model` drops it (step 1's rule: never silently cast). And unlike
`city`, there is no encoder that saves you: frequency-encoding a date asks "how
many other flats were listed on this exact day", which is noise, and target-
encoding it is worse. A date is not a category. It is a **date**, and what you
want from it is *parts* — month, day of week — which is arithmetic, not learning.

That is the gap `transform_features` exists to fill.

In [ ]:
import os, json, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 800

area = rng.uniform(30, 160, size=n).round(1)
price_per_m2 = rng.uniform(2000, 9000, size=n).round(0)
price = (area * price_per_m2).round(0)
year_built = rng.integers(1900, 2024, size=n)
rooms = np.clip((area / 30 + rng.normal(0, 0.5, size=n)).round(), 1, 6)
city = rng.choice(["berlin", "hamburg", "koeln"], size=n)
listed_at = pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 730, size=n), unit="D")

# The real rule: season dominates, weekend listings do well, cheap-per-m2 and
# not-ancient help. Only the last two are visible in the raw numeric columns.
season = np.where(np.isin(listed_at.month, [3, 4, 5]), 1.6,
         np.where(np.isin(listed_at.month, [11, 12, 1]), -1.4, 0.0))
weekend = np.where(listed_at.dayofweek >= 5, 0.9, 0.0)
logit = (season + weekend
         - (price_per_m2 - 5500) / 1500
         - ((2026 - year_built) - 60) / 90
         + rng.normal(0, 0.6, size=n))

df = pd.DataFrame({
    "price": price, "area": area, "rooms": rooms, "year_built": year_built,
    "city": city, "listed_at": listed_at.strftime("%Y-%m-%d"),
    "sold_fast": (logit > 0).astype(int),
})

workdir = tempfile.mkdtemp(prefix="phase3_step3_")
data_path = os.path.join(workdir, "apartments.csv")
df.to_csv(data_path, index=False)
df.head()

**Check yourself:** `price_per_m2` (that is, `price / area`) is also in the
generating rule and is also not a column. Why is that a *weaker* argument for
this tool than `listed_at` is?

> Because the information is already there. A tree can recover `price/area` by
> splitting on `area` and then on `price` — the boundary is diagonal, so it costs
> depth and rows, but nothing is missing. With `listed_at` dropped, the seasonal
> signal is not merely awkward to reach, it is **absent**. Handing a model a
> ratio is a convenience; handing it a parsed date is new information.

## 2. The sandbox, before the tool

`transform_features` runs the agent's code in a subprocess with an import
whitelist (§6). pandas, numpy, scipy and the standard library are in; the model
and sklearn libraries are **out**, with an error that says what to use instead.

That block is not paranoia about malice — it is half of the statelessness
guarantee. Code that cannot import a fitting library cannot fit an estimator.

Only half, though, and the missing half is the interesting one. **Plain pandas
can learn from the data perfectly well.** `pd.qcut` takes quantiles of the column
you hand it; `x / x.mean()` divides by the mean of whatever rows are present;
`rank` ranks within the batch. None of those import anything. And since only the
*code* is stored, replaying it at predict time recomputes those numbers from
whatever rows arrive — so the same row scores differently depending on what it
was batched with. Section 3 deals with that.

In [ ]:
from tabular_agent.transform import transform_features as transform_lib
from tabular_agent.sandbox import SandboxError

blocked = '''
from sklearn.preprocessing import TargetEncoder
def transform_features(df):
    return df
'''
try:
    transform_lib(df, blocked)
except SandboxError as e:
    print(e)

Read the error: it does not just say *no*, it says **where that belongs**. An
agent that gets a bare `ImportError` tries three more spellings; one that is told
`fit_encoding` switches tools.

## 3. The other half: code that reads the whole column

Here is the failure the import block does not catch, and it was found in a real
run — the model wrote `pd.qcut(df["price"], 10)` inside an otherwise excellent
sixty-line transform.

`qcut` cuts a column at its deciles. At training time those deciles come from 800
rows. At predict time they come from whatever arrives — five rows, or one. Same
code, same row, different number.

In [ ]:
import pandas as pd

leaky_bin = '''
import pandas as pd
def transform_features(df):
    df = df.copy()
    df["price_bin"] = pd.qcut(df["price"], 4, labels=False, duplicates="drop")
    return df
'''

# Run it the way training does, then the way serving does.
train_view, _ = transform_lib(df, leaky_bin, check_stability=False)
serve_view, _ = transform_lib(df.head(5), leaky_bin, check_stability=False)

pd.DataFrame({
    "price": df["price"].head(5),
    "bin, batched with 800": train_view["price_bin"].head(5),
    "bin, sent as 5 rows": serve_view["price_bin"],
})

The same flat is in a different bucket depending on who else was in the request.
A model trained on the left column and served the right one is being fed a
feature whose meaning changed underneath it — the same train/serve skew as the
label-encoding trap from step 2, arrived at from a completely different
direction.

Enumerating the guilty pandas calls is unwinnable: any aggregation is one. So the
tool tests the **property** instead of the syntax. It re-runs your code over a
random subset and checks that the rows appearing in both come out identical.
Row-local arithmetic cannot tell the difference; anything batch-dependent
disagrees with itself immediately.

In [ ]:
try:
    transform_lib(df, leaky_bin)      # check_stability=True is the default
except SandboxError as e:
    print(e)

**Check yourself:** `pd.cut(df["price"], bins=[0, 200_000, 500_000, 1e9])` is not
rejected, but `pd.qcut(df["price"], 3)` is. Both produce a binned column. What is
the difference?

> Where the edges come from. `cut` is given its thresholds as literals, so they
> are part of the code and replay identically forever. `qcut` *derives* the
> thresholds from the column in front of it, which makes them a learned parameter
> that is never stored — and a learned parameter that is not stored is exactly
> what this tool cannot keep honest. The rule is not "no binning", it is "no
> numbers that come from the data without being written down".

## 4. The contract: same rows, in the same order

The code must define `transform_features(df)` and return a DataFrame with the
**same row index**. Add columns, replace columns — but never drop, add or reorder
rows.

In [ ]:
good = '''
import pandas as pd
def transform_features(df):
    df = df.copy()
    listed = pd.to_datetime(df["listed_at"])
    df["listed_month"] = listed.dt.month
    df["listed_dow"] = listed.dt.dayofweek
    df["is_weekend"] = (listed.dt.dayofweek >= 5).astype(int)
    df["price_per_m2"] = df["price"] / df["area"]
    df["age"] = 2026 - df["year_built"]
    return df
'''
out, summary = transform_lib(df, good)
print(summary["new_columns"])
out[["listed_at", "listed_month", "listed_dow", "is_weekend", "price_per_m2"]].head(3)

In [ ]:
# Drop rows and the contract fires — with an error written to be read by whoever
# (or whatever) wrote the code.
try:
    transform_lib(df, 'def transform_features(df):\n    return df.query("area > 100")\n')
except SandboxError as e:
    print(e)

Why so strict? The transformed frame is going to sit next to a target column that
was **not** passed through the transform. If the code dropped 200 rows, every
remaining row would be paired with the wrong label — and nothing downstream would
notice, because a shorter frame is a perfectly valid frame. The row index is the
only thing tying features to labels, so it is the thing that gets enforced.

**Check yourself:** the code could still return rows in a different *order* while
keeping the same index values. Why does that not corrupt the labels?

> Because the join back onto the target is by index, not by position. The
> contract checks index *equality* as a set-with-multiplicity, and pandas
> realigns. Dropping a row removes a label's partner; reordering does not.

## 5. Now the tool

Same code, but through the harness: paths in, a compact summary out, artifacts on
disk. `transform_features` writes `tables/<id>.parquet` and the lineage sidecar
next to it — exactly the pattern `fit_encoding` established in step 2.

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import (
    inspect_data, fit_encoding, transform_features, train_tabular_model, finalize_run,
)
from tabular_agent.taskspec import TaskSpec

session = RunSession(
    run_dir=os.path.join(workdir, "run"),
    data_path=data_path,
    spec=TaskSpec(target="sold_fast", n_splits=4),
)

profile = inspect_data(session)
print(profile["task_type"], "|", profile["metric"], "| non-numeric:", profile["non_numeric_columns"])

In [ ]:
t1 = transform_features(session, good)
print(json.dumps(t1, indent=2))

## 6. The design call: the target is not in the frame

The code above received `price`, `area`, `rooms`, `year_built`, `city` — and
**not** `sold_fast`. Reaching for it fails.

In [ ]:
leak = '''
def transform_features(df):
    df = df.copy()
    df["cheat"] = df["sold_fast"]
    return df
'''
try:
    transform_features(session, leak)
except SandboxError as e:
    print(e)

There are two independent reasons for that, and the second is the one that would
actually bite you.

**Reason 1 — leakage.** A column that is a function of the target is the oldest
mistake in the book, and the sandbox's import block does nothing against
`df["cheat"] = df["sold_fast"]`: that is pure pandas. The only robust defence is
not to hand over the target at all. It is the same shape of argument as blocking
sklearn: make the mistake *unexpressible* rather than discouraged.

**Reason 2 — train/serve skew.** At predict time there **is** no target column.
Production sends you a flat to price, not a flat plus the answer. So code that
referenced `sold_fast` would either crash at predict or — worse, if the column
happened to exist under that name — compute something entirely different from
what it computed at training. The frame must be the same shape in both places or
"re-run the code at predict" is a lie.

Reason 2 is why this is enforced rather than merely encouraged: even a
well-intentioned, non-leaky use of the target would break the replay.

**Check yourself:** target encoding *does* use the target, and we ship it. Why is
that allowed when this is not?

> Because `fit_encoding(method="target")` splits the two jobs: it trains on
> out-of-fold values and serves a mapping fitted on all rows, so the served
> transform needs only the *category*, never a target. Free code has no such
> machinery — it would store one column and pretend it was reproducible.

## 7. What gets stored: the code, not the columns

Open the lineage sidecar. There is no `price_per_m2` column in it — there is the
program that makes one.

In [ ]:
lineage = json.load(open(os.path.join(session.run_dir, "tables", "t1.pipeline.json")))
step = lineage["steps"][0]
print("step type :", step["type"])
print("func      :", step["func_name"])
print("code      :")
print(step["code"])

This is the difference between a feature *store* and a feature *pipeline*. If we
stored the produced columns, then serving a new flat would mean looking up a
value that was never computed for it. Storing the code means a new row goes
through the identical arithmetic — reproducible by construction, and impossible
to skew.

It also means the journal can show what the agent *tried*, not just what it got:

In [ ]:
from tabular_agent.harness import journal

for t in journal.load(session.run_dir)["tables"]:
    print(t["id"], "<-", t["source_table"], "|", t["method"], "|", t["new_columns"])

## 8. Stacking: encode, then transform on top

Tables chain. `city` is a nominal string — a genuine job for `fit_encoding` —
so encode it into `t2`, then derive a feature *from the encoded column* into
`t3`. Each table keeps the full ordered lineage of everything behind it.

In [ ]:
t2 = fit_encoding(session, ["city"], method="frequency", table=t1["table"])

stacked = '''
def transform_features(df):
    df = df.copy()
    df["price_per_m2_vs_city_freq"] = df["price_per_m2"] * df["city_freq"]
    return df
'''
t3 = transform_features(session, stacked, table=t2["table"])
print(t3["table"], "<-", t3["source_table"], "|", t3["new_columns"])

steps = json.load(open(os.path.join(session.run_dir, "tables", "t3.pipeline.json")))["steps"]
print("lineage of t3:", [s["type"] for s in steps])

`["transform", "encode", "transform"]` — in the order they must replay. That
ordering is not bookkeeping: `city_freq` has to exist before the second transform
can multiply by it, at predict time just as much as here.

## 9. Does it help?

Four models: the trivial baseline, the raw table, the derived features, and the
full stack. The metric is logloss, so **lower is better**.

In [ ]:
base = train_tabular_model(session, model_type="baseline")
raw = train_tabular_model(session, model_type="lgbm")
derived = train_tabular_model(session, table=t1["table"], model_type="lgbm")
full = train_tabular_model(session, table=t3["table"], model_type="lgbm")

for name, r in [("baseline", base), ("raw", raw), ("+derived", derived), ("+city", full)]:
    print(f"{name:9} {r['cv_score']:.4f}  +/- {r['fold_spread']:.4f}  ({r['n_features']} features)")

Look at `raw` against `baseline`. A gradient-boosted model, 800 rows, four
perfectly good numeric columns — and it lands **level with predicting the base
rate**, well inside its own fold spread. Not because the model is weak, but
because the column that decides the outcome sits on disk as a string and was
dropped. That is what "we dropped the non-numeric columns" looks like when one of
them mattered.

`+derived` fixes it, and by a lot: ~0.69 to ~0.50, roughly ten fold-spreads. Same
rows, same model, same hyper-parameters — the only change is that the date got
parsed.

`+city` is the one to be careful with. It adds a frequency encoding and a product
on top, and lands within a hair of `+derived` — a gap far smaller than
`fold_spread`. The honest conclusion is "no evidence it helps", and the harness
exists to make you say that rather than quietly keep whichever number came out
lower.

**Check yourself:** `baseline` has a fold spread of ~0.0004 and `raw` ~0.04, a
hundred times larger. What is that telling you?

> The baseline predicts one constant, so it scores almost identically on every
> fold — its spread is measuring the folds, not the model. A real model's spread
> is measuring how much its *fit* changes with the training rows. Large spread
> with no gain over baseline, as here, is the signature of a model fitting noise:
> it finds something different in each fold because there is nothing to find.

## 10. Finalize — the code has to run again on raw rows

The payoff. `finalize_run` packages the lineage of whichever table the winning
model trained on. When we then predict from **raw** rows — no derived columns, no
target — the bundle re-runs the stored code in the sandbox to rebuild them.

In [ ]:
final = finalize_run(session)
print("best:", final["best_experiment_id"], "| cv:", round(final["cv_score"], 4),
      "| baseline:", round(final["baseline_score"], 4))

In [ ]:
from tabular_agent.io import load_table
from tabular_agent.pipeline import load_bundle, predict_bundle

bundle = load_bundle(os.path.join(session.run_dir, final["bundle_dir"]))
print("pipeline steps:", [s["type"] for s in bundle["pipeline"].to_dict()["steps"]])
print("features      :", bundle["feature_names"])

In [ ]:
# Exactly what production sends: the original columns, no target.
raw_rows = load_table(data_path).drop(columns=["sold_fast"]).head(5)
print(raw_rows.to_string(index=False))
print()
print("predictions:", predict_bundle(bundle, raw_rows).round(3))

Those five rows never had a `price_per_m2` column. The bundle made one, by
running the same string of Python the agent wrote during the run — in the same
sandbox, with the same import block. There is no separate serving path to keep in
sync, which is the only reason this is safe to automate.

**Check yourself:** the sandbox spawns a subprocess per call. What does that cost
you at predict time, and when would it matter?

> A process launch per `transform` call — milliseconds, invisible for a batch of
> a million rows, but real if you are serving single requests at high QPS. The
> fix is batching, not removing the sandbox; and note that a bundle with no code
> step never imports the sandbox at all (`Pipeline.transform` imports it lazily).

## 11. Five tools

Every tool is a hole in the "leakage-safe by construction" claim, so the surface
is asserted in the tests rather than merely documented. `transform_features` is
the widest hole yet — arbitrary Python — which is why it comes with three
independent constraints: the import block (cannot learn), the row-index contract
(cannot misalign), and the missing target column (cannot leak, cannot skew).

In [ ]:
from tabular_agent.harness.agent import build_agent, DEFAULT_MODEL, DEFAULT_GATEWAY_MODEL

agent = build_agent()
print("tools:", sorted(agent._function_toolset.tools))
print("tool calls in flight at once:", agent._concurrency_limiter.max_running)

That last line is new, and it is a bug fix rather than a feature. A model may put
several tool calls in **one** message, and PydanticAI will happily run them
concurrently. Our id claims (`next_table_id`, `next_experiment_id`) read the
journal and write it back with no lock, so two parallel `transform_features`
calls can both decide they are `t4` and race on the same parquet file. Two
parallel trainings colliding on `e7` would be worse: the artifacts would be
overwritten *silently*, and the leaderboard would show two rows pointing at one
directory.

`max_concurrency=1` serialises them. It costs nothing here — every tool call is
seconds of local compute, not a network wait — and it is only half a fix: it is
per-`Agent`, so two processes pointed at the same run dir would still collide.
The real fix is an exclusive-create id claim, which is still on the list.

## 12. Live run

Now hand it over. Same tools, same run dir shape — the only difference is that
the code in the transform step will be written by the model.

Key comes from `config.env` (`REQUESTY_API_KEY=...`), which is gitignored. Use
`await`, not `run_sync`: a notebook already has an event loop.

In [ ]:
from tabular_agent.harness.agent import Deps, gateway_model

model = gateway_model(DEFAULT_GATEWAY_MODEL)   # tensorx/deepseek-v4-flash
print("model:", DEFAULT_GATEWAY_MODEL, "| default (needs an Anthropic key):", DEFAULT_MODEL)

In [ ]:
fresh = RunSession(
    run_dir=os.path.join(workdir, "agent_run"),
    data_path=data_path,
    spec=TaskSpec(target="sold_fast", n_splits=4),
)

result = await build_agent(model).run(
    "Build the best model you can for this dataset, then finalize the run. "
    "The columns are about apartments being sold.",
    deps=Deps(session=fresh),
)
print(result.output)

In [ ]:
# The record, not the model's account of itself.
j = journal.load(fresh.run_dir)
for t in j.get("tables", []):
    print(t["id"], "|", t["method"], "|", t["new_columns"])
    if t["method"] == "transform":
        print(t["code"])
print()
for r in j["leaderboard"]:
    print(f"{r['id']:4} {r['model_type']:10} {r['cv_score']:8.4f}  {os.path.basename(r['source_table'])}")
print("\nbest:", j["best"])

The `code` field printed above is the thing to read carefully. It is the agent's
reasoning made *executable and permanent* — not a sentence in its summary saying
"I engineered some ratio features", but the exact program, stored in the run and
re-run at predict time. If it wrote something silly, you can see it. If it wrote
something good, you keep it for free.

Three things worth checking in your own run:

- **Did it parse `listed_at`?** That is the whole dataset. A run that never
  transforms the date will sit level with the baseline no matter how many models
  it tries, and the leaderboard will show it flat.
- **How much did it write?** When this notebook was written, the model produced
  about twenty columns in one call: the date parts (right), plus `price_sq`,
  `area_sq`, `price_x_rooms`, `age_sq`, `log_price`. Squares and products are
  free noise for a tree — it can already split anywhere on `price`, so `price_sq`
  adds a column and no information. Nothing is *broken* by that; it just spends
  the model's attention. This is the shape of the "harness enforces safe, not
  smart" point from step 2, and it is an argument for the §19 stop rule.
- **Did it try to hand-build an encoder?** It cannot import sklearn, but nothing
  stops it computing a `groupby` mean in pandas — except that the frame it gets
  has no target column, so the leaky version is unreachable. What it *can* build
  is a group mean over a feature, which is a real (if fold-blind) statistic.
  Watch for it: that belongs in `fit_encoding`.

**Next:** `blend_models` and `predict` are the two tools still missing, and then
§19's compaction and harness-owned stop — the run above will happily keep
experimenting long past the point where the score stops moving.